## Code-regression preprocessing & EDA script

1. Stream dataset from huggingface. 
2. Filter to CodeNet samples
3. 

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import QuantileTransformer

In [8]:
# Run once. Should take ~1min to stream
# df = pl.read_parquet('hf://datasets/akhauriyash/Code-Regression/data.parquet')
# df.write_parquet("./raw_code_regression_data.parquet")

# Read cache
df = pl.read_parquet("./raw_code_regression_data.parquet")
df

identifier,space,input,target,metric_type,metadata
str,str,str,f64,str,str
"""APPS_11588""","""APPS""","""Given an array of integers arr…",6036.0,"""memory_bytes""","""{'question_id': '0291', 'solut…"
"""APPS_85535""","""APPS""","""Complete the method which acce…",5354.0,"""memory_bytes""","""{'question_id': '4146', 'solut…"
"""APPS_85418""","""APPS""","""A special type of prime is gen…",7244.0,"""memory_bytes""","""{'question_id': '4141', 'solut…"
"""APPS_93858""","""APPS""","""Complete the function/method s…",6591.0,"""memory_bytes""","""{'question_id': '4510', 'solut…"
"""APPS_37440""","""APPS""","""Coach Khaled is a swag teacher…",13235.0,"""memory_bytes""","""{'question_id': '1284', 'solut…"
…,…,…,…,…,…
"""CDSS_136707""","""CDSS""","""#include <stdio.h> #define N 1…",636.0,"""memory_bytes""","""{'s_id': 's409024552', 'p_id':…"
"""CDSS_11873426""","""CDSS""","""X, A = map(int, input().split(…",2940.0,"""memory_bytes""","""{'s_id': 's422507313', 'p_id':…"
"""CDSS_10789316""","""CDSS""","""from collections import defaul…",42600.0,"""memory_bytes""","""{'s_id': 's804190988', 'p_id':…"


In [11]:
preliminary_df = (
    df
    .filter(pl.col("space").eq("CDSS"))
    .with_columns([
        pl.col("metadata").str.extract(r"'p_id': '([^']+)'").alias("p_id"),
        pl.col("metadata").str.extract(r"'u_id': '([^']+)'").alias("u_id"),
        pl.col("metadata").str.extract(r"'language': '([^']+)'").alias("language"),
        pl.col("metadata").str.extract(r"'code_size': '(\d+)'").cast(pl.Int32).alias("code_size"),
        pl.col("metadata").str.extract(r"'cpu_time': '(\d+)'").cast(pl.Int32).alias("cpu_time"),
    ])
    .rename({"identifier": "id", "target": "memory_bytes"})
    # code_size is just the code length
    .select(["id", "input", "memory_bytes", "p_id", "u_id", "language", "code_size", "cpu_time"])
)
preliminary_df

id,input,memory_bytes,p_id,u_id,language,code_size,cpu_time
str,str,f64,str,str,str,i32,i32
"""CDSS_6335723""","""#include<cstdio> #include<iost…",256.0,"""p03250""","""u426572476""","""C++""",332,1
"""CDSS_12494010""","""R=int(input()) if R<1200: …",2940.0,"""p03288""","""u349444371""","""Python""",95,17
"""CDSS_3079361""","""#include <bits/stdc++.h> using…",3636.0,"""p02694""","""u445619807""","""C++""",230,5
"""CDSS_4803396""","""#include <bits/stdc++.h> using…",384.0,"""p02949""","""u166378830""","""C++""",971,73
"""CDSS_4851225""","""//minamoto #include<bits/stdc+…",896.0,"""p02954""","""u561765782""","""C++""",766,11
…,…,…,…,…,…,…,…
"""CDSS_136707""","""#include <stdio.h> #define N 1…",636.0,"""p02238""","""u610059172""","""C""",512,0
"""CDSS_11873426""","""X, A = map(int, input().split(…",2940.0,"""p02999""","""u338225045""","""Python""",68,17
"""CDSS_10789316""","""from collections import defaul…",42600.0,"""p02689""","""u975116284""","""Python""",552,436


In [ ]:
lang_counts = (
    preliminary_df.group_by("language")
    .agg(pl.len().alias("count"))
    .sort("count")
)
px.bar(lang_counts, x="count", y="language", orientation="h", height=700)

## Token distribution

In [ ]:
lang_code_size = (
    preliminary_df.group_by("language")
    .agg(pl.col("code_size").mean().alias("mean_code_size"))
    .sort("mean_code_size")
)
px.bar(lang_code_size, x="mean_code_size", y="language", orientation="h", height=700)

## Responder distribution & processing

In [ ]:
lang_order = (
    preliminary_df
    .with_columns(pl.col("memory_bytes").log1p().alias("log1p_mem"))
    .group_by("language")
    .agg([pl.col("log1p_mem").mean().alias("mean_log1p"), pl.len().alias("count")])
    .filter(pl.col("count") >= 500)
    .sort("mean_log1p")
)
languages = lang_order["language"].to_list()
rng = np.random.default_rng(0)

lang_data = {k[0]: v["memory_bytes"].to_numpy()
    for k, v in preliminary_df.select(["language","memory_bytes"]).partition_by("language", as_dict=True).items()}

fig = go.Figure()
for i, lang in enumerate(languages):
    data = np.log1p(lang_data[lang])
    sample = rng.choice(data, size=min(len(data), 20_000), replace=False)
    kde = stats.gaussian_kde(sample, bw_method=0.15)
    xlo, xhi = np.percentile(data, [0.1, 99.9])
    x = np.linspace(xlo, xhi, 400)
    y = kde(x) / kde(x).max() * 0.9
    color = px.colors.qualitative.Plotly[i % 10]
    fig.add_trace(go.Scatter(x=x, y=np.full_like(x, i), mode="lines",
                             line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=x, y=i+y, name=lang, fill="tonexty",
                             fillcolor=color, line=dict(color=color, width=1)))

fig.update_layout(
    height=100 + len(languages)*30,
    margin=dict(l=120),
    xaxis_title="log1p(memory_bytes)",
    yaxis=dict(tickmode="array", tickvals=list(range(len(languages))), ticktext=languages),
    showlegend=False,
)
fig

In [48]:
lang_stats = (
    preliminary_df.group_by("language")
    .agg([pl.len().alias("count"), pl.col("memory_bytes").mean().alias("mean_mb")])
    .filter(pl.col("count") >= 500).sort("mean_mb")
)
languages = lang_stats["language"].to_list()
lang_counts = dict(zip(lang_stats["language"].to_list(), lang_stats["count"].to_list()))

# Partition once to avoid repeated full-table scans
lang_data = {k[0]: v["memory_bytes"].to_numpy()
    for k, v in preliminary_df.select(["language","memory_bytes"]).partition_by("language", as_dict=True).items()}

y_all = preliminary_df["memory_bytes"].to_numpy().reshape(-1, 1)
qt_global = QuantileTransformer(n_quantiles=10_000, output_distribution="normal", subsample=500_000, random_state=0)
qt_global.fit(y_all)

qt_per_lang = {}
for lang in languages:
    y = lang_data[lang].reshape(-1, 1)
    n = len(y)
    qt = QuantileTransformer(n_quantiles=min(1000, n), output_distribution="normal",
                             subsample=min(100_000, n), random_state=0)
    qt.fit(y)
    qt_per_lang[lang] = qt

rng = np.random.default_rng(0)
ytick_labels = [f"{l}  (n={lang_counts[l]:,})" for l in languages]

fig = make_subplots(rows=2, cols=2, row_heights=[0.85, 0.15],
    subplot_titles=["Global QT", "Per-language QT", "", ""], vertical_spacing=0.05)

global_sample, perlang_sample = [], []
for i, lang in enumerate(languages):
    y_lang = lang_data[lang].reshape(-1, 1)
    color = px.colors.qualitative.Plotly[i % 10]
    g_data = qt_global.transform(y_lang).ravel()
    p_data = qt_per_lang[lang].transform(y_lang).ravel()
    n_sub = min(len(g_data), 5_000)
    global_sample.append(rng.choice(g_data, n_sub, replace=False))
    perlang_sample.append(rng.choice(p_data, n_sub, replace=False))

    for col, data in [(1, g_data), (2, p_data)]:
        sample = rng.choice(data, size=min(len(data), 20_000), replace=False)
        kde = stats.gaussian_kde(sample, bw_method=0.2)
        xlo, xhi = np.percentile(data, [0.5, 99.5])
        x = np.linspace(xlo, xhi, 300)
        y = kde(x) / kde(x).max() * 0.9
        fig.add_trace(go.Scatter(x=x, y=np.full_like(x, i), mode="lines",
                                 line=dict(width=0), showlegend=False, hoverinfo="skip"), row=1, col=col)
        fig.add_trace(go.Scatter(x=x, y=i+y, fill="tonexty", fillcolor=color,
                                 line=dict(color=color, width=1), showlegend=False), row=1, col=col)

fig.add_trace(go.Histogram(x=np.concatenate(global_sample), nbinsx=150,
                           showlegend=False, marker_color="steelblue"), row=2, col=1)
fig.add_trace(go.Histogram(x=np.concatenate(perlang_sample), nbinsx=150,
                           showlegend=False, marker_color="steelblue"), row=2, col=2)
fig.update_layout(
    height=250 + len(languages)*30,
    margin=dict(l=180),
    yaxis=dict(tickmode="array", tickvals=list(range(len(languages))), ticktext=ytick_labels),
    yaxis2=dict(tickmode="array", tickvals=list(range(len(languages))), ticktext=ytick_labels),
)
fig

In [49]:
lang_arr = preliminary_df["language"].to_numpy()
mb_arr = preliminary_df["memory_bytes"].to_numpy()
y_qn = np.empty(len(preliminary_df))

for lang in preliminary_df["language"].unique().to_list():
    mask = lang_arr == lang
    y_lang = mb_arr[mask].reshape(-1, 1)
    qt = qt_per_lang.get(lang)
    if qt is None:
        n = len(y_lang)
        qt = QuantileTransformer(n_quantiles=min(500, n), output_distribution="normal",
                                 subsample=n, random_state=0)
        qt.fit(y_lang)
    y_qn[mask] = qt.transform(y_lang).ravel()

processed_df = preliminary_df.with_columns(pl.Series("memory_bytes_qn", y_qn))
processed_df.write_parquet("./processed_codenet_data.parquet")
processed_df

id,input,memory_bytes,p_id,u_id,language,code_size,cpu_time,memory_bytes_qn
str,str,f64,str,str,str,i32,i32,f64
"""CDSS_6335723""","""#include<cstdio> #include<iost…",256.0,"""p03250""","""u426572476""","""C++""",332,1,-0.837338
"""CDSS_12494010""","""R=int(input()) if R<1200: …",2940.0,"""p03288""","""u349444371""","""Python""",95,17,-1.132898
"""CDSS_3079361""","""#include <bits/stdc++.h> using…",3636.0,"""p02694""","""u445619807""","""C++""",230,5,0.610484
"""CDSS_4803396""","""#include <bits/stdc++.h> using…",384.0,"""p02949""","""u166378830""","""C++""",971,73,-0.266584
"""CDSS_4851225""","""//minamoto #include<bits/stdc+…",896.0,"""p02954""","""u561765782""","""C++""",766,11,-0.116941
…,…,…,…,…,…,…,…,…
"""CDSS_136707""","""#include <stdio.h> #define N 1…",636.0,"""p02238""","""u610059172""","""C""",512,0,0.116941
"""CDSS_11873426""","""X, A = map(int, input().split(…",2940.0,"""p02999""","""u338225045""","""Python""",68,17,-1.132898
"""CDSS_10789316""","""from collections import defaul…",42600.0,"""p02689""","""u975116284""","""Python""",552,436,1.108798


In [ ]:
# Zooming out, left-tail problem's not as bad as it seems!
processed_df.sort("memory_bytes_qn").hvplot.hist("memory_bytes_qn", bins=30)

:Histogram   [memory_bytes_qn]   (Count)